# 04 BM25

## Objetivos

* Implementar BM25 y comparar su ranking con el de TF-IDF + coseno sobre la misma colección de documentos.
* Calcular el aporte de cada término en BM25, identificar qué controlan $k_1$ y $b$, y justificar un cambio de posición usando las estadísticas de los documentos.

**1. Carga el corpus de Project Gutenberg**

Aplicamos la misma tokenización del anterior notebook, ademas de precalcular dos IDF distintos ya que se asignan pesos diferentes a los terminos en TF-IDF y BM25

In [1]:
import re
import unicodedata
from collections import Counter

import nltk
import numpy as np
import pandas as pd
from IPython.display import display
from nltk.corpus import gutenberg
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('gutenberg', quiet=True)
identificadores = gutenberg.fileids()

patron_token = re.compile(r"\w+(?:'\w+)*", flags=re.UNICODE)

def normalizar_tokenizar(texto):
    texto = texto.lower()
    texto = texto.replace('\u2019', "'")
    texto = texto.replace('\u2018', "'")
    texto = texto.replace('_', ' ')
    texto = unicodedata.normalize('NFC', texto)
    return patron_token.findall(texto)

def titulo_del_texto(texto, doc_id):
    for linea in texto[:2000].splitlines():
        linea = linea.strip()
        if linea.startswith('['):
            cierre = linea.find(']')
            if cierre > 1:
                titulo = linea[1:cierre]
                fin = titulo.lower().find(' by ')
                if fin >= 0:
                    titulo = titulo[:fin]
                return titulo.strip()
    raise ValueError('Sin titulo: ' + doc_id)

filas_corpus = []
tf_por_documento = {}
df_por_termino = Counter()
for doc_id in identificadores:
    texto = gutenberg.raw(doc_id)
    tokens = normalizar_tokenizar(texto)
    titulo = titulo_del_texto(texto, doc_id)
    filas_corpus.append({
        'doc_id': doc_id, 'Titulo': titulo, 'Texto': texto, 'tokens': tokens,
    })
    conteos = Counter(tokens)
    tf_por_documento[doc_id] = conteos
    df_por_termino.update(conteos.keys())

corpus = pd.DataFrame(filas_corpus)
N = len(corpus)
longitudes = {}
for fila in filas_corpus:
    longitudes[fila['doc_id']] = len(fila['tokens'])
longitud_media = sum(longitudes.values()) / N

# IDF de TF-IDF
idf_tfidf = {}
# IDF no negativo de BM25
idf_bm25 = {}
for termino, df in df_por_termino.items():
    idf_tfidf[termino] = float(np.log(N / df) + 1)
    idf_bm25[termino] = float(np.log(1 + (N - df + 0.5) / (df + 0.5)))

display(corpus[['doc_id', 'Titulo', 'Texto', 'tokens']])
print('Documentos:', N)
print('Vocabulario:', len(df_por_termino))
print('Longitud media:', round(longitud_media, 2))

,doc_id,Titulo,Texto,tokens
0,austen-emma.txt,Emma,[Emma by Jane Austen 1816]\n\nVOLUME I\n\nCHAP...,"[emma, by, jane, austen, 1816, volume, i, chap..."
1,austen-persuasion.txt,Persuasion,[Persuasion by Jane Austen 1818]\n\n\nChapter ...,"[persuasion, by, jane, austen, 1818, chapter, ..."
2,austen-sense.txt,Sense and Sensibility,[Sense and Sensibility by Jane Austen 1811]\n\...,"[sense, and, sensibility, by, jane, austen, 18..."
3,bible-kjv.txt,The King James Bible,[The King James Bible]\n\nThe Old Testament of...,"[the, king, james, bible, the, old, testament,..."
4,blake-poems.txt,Poems,[Poems by William Blake 1789]\n\n \nSONGS OF I...,"[poems, by, william, blake, 1789, songs, of, i..."
5,bryant-stories.txt,Stories to Tell to Children,[Stories to Tell to Children by Sara Cone Brya...,"[stories, to, tell, to, children, by, sara, co..."
6,burgess-busterbrown.txt,The Adventures of Buster Bear,[The Adventures of Buster Bear by Thornton W. ...,"[the, adventures, of, buster, bear, by, thornt..."
7,carroll-alice.txt,Alice's Adventures in Wonderland,[Alice's Adventures in Wonderland by Lewis Car...,"[alice's, adventures, in, wonderland, by, lewi..."
8,chesterton-ball.txt,The Ball and The Cross,[The Ball and The Cross by G.K. Chesterton 190...,"[the, ball, and, the, cross, by, g, k, chester..."
9,chesterton-brown.txt,The Wisdom of Father Brown,[The Wisdom of Father Brown by G. K. Chesterto...,"[the, wisdom, of, father, brown, by, g, k, che..."


Documentos: 18
Vocabulario: 43824
Longitud media: 121224.33


**2. Benchmarking**

Utilizamos $k_1=1.2$, $b=0.75$, además de construir la matriz dispersa TF-IDF a partir de los tokens ya preparados anteriormente, comprobando si el IDF coincide con el calculado antes. Luego de esto, definimos la puntuación BM25 donde cada termino aporta segun la frecuencia en el libro, asi como la rareza y la longitud del libro con respecto a la media. Dejamos fuera los scores cero y desempatamos con el identificador del documento

In [2]:
def conservar_tokens(tokens):
    return tokens

terminos = sorted(df_por_termino)
columna_de = {}
for columna, termino in enumerate(terminos):
    columna_de[termino] = columna

documentos_procesados = []
for fila in filas_corpus:
    documentos_procesados.append(fila['tokens'])

vectorizador = TfidfVectorizer(
    analyzer=conservar_tokens,
    vocabulary=columna_de,
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=False,
    norm=None,
)
matriz_tfidf = vectorizador.fit_transform(documentos_procesados).tocsr()
idf_esperado = []
for termino in terminos:
    idf_esperado.append(idf_tfidf[termino])
if not np.allclose(vectorizador.idf_, idf_esperado):
    raise ValueError('No coincide IDF TF-IDF')
print('Matriz dispersa TF-IDF:', matriz_tfidf.shape)

k1_base = 1.2
b_base = 0.75

def aporte_bm25(termino, doc_id, k1, b):
    f = tf_por_documento[doc_id][termino]
    if f == 0:
        return 0.0
    ajuste = k1 * (1 - b + b * longitudes[doc_id] / longitud_media)
    return idf_bm25[termino] * f * (k1 + 1) / (f + ajuste)

def scores_bm25(tokens_q, k1, b):
    scores = np.zeros(N)
    vistos = set()
    for termino in tokens_q:
        if termino in vistos:
            continue
        vistos.add(termino)
        if termino not in idf_bm25:
            continue
        for i, doc_id in enumerate(identificadores):
            scores[i] += aporte_bm25(termino, doc_id, k1, b)
    return scores

def ordenar_positivos(scores):
    indices = []
    for i in range(N):
        if scores[i] > 0:
            indices.append(i)

    def criterio(i):
        return (-float(scores[i]), str(identificadores[i]))

    return sorted(indices, key=criterio)

Matriz dispersa TF-IDF: (18, 43824)


Luego ejecutamos ocho consultas de distinta longitud y frecuencia, incluyendo una palabra que no existe en el vocabulario. Para cada una aplica el mismo preprocesamiento que se aplicó al corpus, calculando la similitud TF-IDF con coseno y el score BM25, tambien ordenando por separado los resultados positivos. Imprimimos así el DF de cada término con diez documentos por método. La consulta sin vocabulario sirve para comprobar las listas vacías

In [3]:
consultas = [
    ('the', 'Buscar un termino general presente en muchos libros.'),
    ('whale', 'Buscar menciones de ballenas.'),
    ('alice rabbit', 'Buscar la historia de Alicia y el conejo.'),
    ('emma marriage love', 'Buscar una novela de Emma y relaciones amorosas.'),
    ('king god israel', 'Buscar pasajes religiosos sobre reyes e Israel.'),
    ('sea ship whale water', 'Buscar relatos maritimos sobre barcos y ballenas.'),
    ('father brown detective murder mystery', 'Buscar relatos de investigacion criminal.'),
    ('zzzxqnotaword', 'Comprobar una consulta sin palabras del corpus.'),
]

casos = []
for consulta, necesidad in consultas:
    tokens_q = normalizar_tokenizar(consulta)
    q_tfidf = vectorizador.transform([tokens_q])
    cosenos = cosine_similarity(q_tfidf, matriz_tfidf).ravel()
    bm25 = scores_bm25(tokens_q, k1_base, b_base)
    orden_tfidf = ordenar_positivos(cosenos)
    orden_bm25 = ordenar_positivos(bm25)
    casos.append({
        'consulta': consulta, 'necesidad': necesidad, 'tokens': tokens_q,
        'q_tfidf': q_tfidf, 'cosenos': cosenos, 'bm25': bm25,
        'orden_tfidf': orden_tfidf, 'orden_bm25': orden_bm25,
    })

    print('\nConsulta:', consulta, '-', necesidad)
    for termino in tokens_q:
        print('  df(', termino, '):', df_por_termino[termino])
    print('TF-IDF:')
    for posicion, i in enumerate(orden_tfidf[:10], start=1):
        print(posicion, identificadores[i], round(float(cosenos[i]), 6))
    print('BM25:')
    for posicion, i in enumerate(orden_bm25[:10], start=1):
        print(posicion, identificadores[i], round(float(bm25[i]), 6))
    if len(orden_tfidf) == 0:
        if len(orden_bm25) > 0:
            raise ValueError('Consulta vacia no coincide')

print('\nConsultas procesadas:', len(casos))


Consulta: the - Buscar un termino general presente en muchos libros.
  df( the ): 18
TF-IDF:
1 whitman-leaves.txt 0.72954
2 melville-moby_dick.txt 0.676281
3 bryant-stories.txt 0.668469
4 chesterton-brown.txt 0.639008
5 blake-poems.txt 0.615449
6 bible-kjv.txt 0.606733
7 chesterton-thursday.txt 0.594141
8 chesterton-ball.txt 0.587963
9 carroll-alice.txt 0.561944
10 edgeworth-parents.txt 0.505853
BM25:
1 bible-kjv.txt 0.058664
2 whitman-leaves.txt 0.058663
3 melville-moby_dick.txt 0.058662
4 chesterton-brown.txt 0.05866
5 chesterton-ball.txt 0.058659
6 bryant-stories.txt 0.058659
7 edgeworth-parents.txt 0.058658
8 chesterton-thursday.txt 0.058658
9 austen-persuasion.txt 0.058654
10 austen-emma.txt 0.058653

Consulta: whale - Buscar menciones de ballenas.
  df( whale ): 6
TF-IDF:
1 melville-moby_dick.txt 0.10828
2 bryant-stories.txt 0.006502
3 shakespeare-hamlet.txt 0.00154
4 whitman-leaves.txt 0.000757
5 chesterton-ball.txt 0.000495
6 bible-kjv.txt 4e-05
BM25:
1 melville-moby_dick.txt 

Existen términos muy frecuentes (*the*, *water*, *god*, *sea*), poco frecuentes (*whale*, *alice*, *emma*, *detective*, *israel*) y longitudes de consulta de 1 a 5 términos. Se tiene que un libro puntúa solo si contiene al menos una palara de la consulta

**3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |


Realizamos la tabla propuesta, teniendo en cuenta que los scores de coseno y BM25 tienen escalas distintas, por lo que comparamos las posiciones de cada ranking y no los valores entre métodos

In [4]:
resumen = []
for caso in casos:
    top_tfidf = caso['orden_tfidf'][:10]
    top_bm25 = caso['orden_bm25'][:10]
    filas = []
    n_filas = max(len(top_tfidf), len(top_bm25))
    for pos in range(n_filas):
        fila = {'Posicion': pos + 1, 'ID y titulo TF-IDF': '', 'Coseno': None,
                'ID y titulo BM25': '', 'Score BM25': None}
        if pos < len(top_tfidf):
            i = top_tfidf[pos]
            fila['ID y titulo TF-IDF'] = identificadores[i] + ' - ' + filas_corpus[i]['Titulo']
            fila['Coseno'] = float(caso['cosenos'][i])
        if pos < len(top_bm25):
            i = top_bm25[pos]
            fila['ID y titulo BM25'] = identificadores[i] + ' - ' + filas_corpus[i]['Titulo']
            fila['Score BM25'] = float(caso['bm25'][i])
        filas.append(fila)

    ids_tf = set()
    ids_bm = set()
    for i in top_tfidf:
        ids_tf.add(identificadores[i])
    for i in top_bm25:
        ids_bm.add(identificadores[i])
    primero_tf = identificadores[top_tfidf[0]] if top_tfidf else '(vacio)'
    primero_bm = identificadores[top_bm25[0]] if top_bm25 else '(vacio)'
    resumen.append({
        'Consulta': caso['consulta'], 'Tamano TF-IDF': len(top_tfidf),
        'Tamano BM25': len(top_bm25), 'IDs compartidos': len(ids_tf & ids_bm),
        'Primero TF-IDF': primero_tf, 'Primero BM25': primero_bm,
    })
    print('\nConsulta:', caso['consulta'])
    columnas_tabla = ['Posicion', 'ID y titulo TF-IDF', 'Coseno',
                      'ID y titulo BM25', 'Score BM25']
    display(pd.DataFrame(filas, columns=columnas_tabla).round({
        'Coseno': 6, 'Score BM25': 6,
    }))

display(pd.DataFrame(resumen))


Consulta: the


,Posicion,ID y titulo TF-IDF,Coseno,ID y titulo BM25,Score BM25
0,1,whitman-leaves.txt - Leaves of Grass,0.729540,bible-kjv.txt - The King James Bible,0.058664
1,2,melville-moby_dick.txt - Moby Dick,0.676281,whitman-leaves.txt - Leaves of Grass,0.058663
2,3,bryant-stories.txt - Stories to Tell to Children,0.668469,melville-moby_dick.txt - Moby Dick,0.058662
3,4,chesterton-brown.txt - The Wisdom of Father Brown,0.639008,chesterton-brown.txt - The Wisdom of Father Brown,0.058660
4,5,blake-poems.txt - Poems,0.615449,chesterton-ball.txt - The Ball and The Cross,0.058659
5,6,bible-kjv.txt - The King James Bible,0.606733,bryant-stories.txt - Stories to Tell to Children,0.058659
6,7,chesterton-thursday.txt - The Man Who Was Thur...,0.594141,"edgeworth-parents.txt - The Parent's Assistant,",0.058658
7,8,chesterton-ball.txt - The Ball and The Cross,0.587963,chesterton-thursday.txt - The Man Who Was Thur...,0.058658
8,9,carroll-alice.txt - Alice's Adventures in Wond...,0.561944,austen-persuasion.txt - Persuasion,0.058654
9,10,"edgeworth-parents.txt - The Parent's Assistant,",0.505853,austen-emma.txt - Emma,0.058653



Consulta: whale


,Posicion,ID y titulo TF-IDF,Coseno,ID y titulo BM25,Score BM25
0,1,melville-moby_dick.txt - Moby Dick,0.108280,melville-moby_dick.txt - Moby Dick,2.355725
1,2,bryant-stories.txt - Stories to Tell to Children,0.006502,bryant-stories.txt - Stories to Tell to Children,2.268508
2,3,shakespeare-hamlet.txt - The Tragedie of Hamlet,0.001540,whitman-leaves.txt - Leaves of Grass,1.896003
3,4,whitman-leaves.txt - Leaves of Grass,0.000757,shakespeare-hamlet.txt - The Tragedie of Hamlet,1.872497
4,5,chesterton-ball.txt - The Ball and The Cross,0.000495,chesterton-ball.txt - The Ball and The Cross,1.621518
5,6,bible-kjv.txt - The King James Bible,0.000040,bible-kjv.txt - The King James Bible,0.547050



Consulta: alice rabbit


,Posicion,ID y titulo TF-IDF,Coseno,ID y titulo BM25,Score BM25
0,1,carroll-alice.txt - Alice's Adventures in Wond...,0.315076,carroll-alice.txt - Alice's Adventures in Wond...,6.051943
1,2,burgess-busterbrown.txt - The Adventures of Bu...,0.007630,chesterton-thursday.txt - The Man Who Was Thur...,3.509562
2,3,bryant-stories.txt - Stories to Tell to Children,0.002442,burgess-busterbrown.txt - The Adventures of Bu...,2.273300
3,4,chesterton-ball.txt - The Ball and The Cross,0.000595,bryant-stories.txt - Stories to Tell to Children,2.217046
4,5,chesterton-thursday.txt - The Man Who Was Thur...,0.000571,"edgeworth-parents.txt - The Parent's Assistant,",2.099567
5,6,chesterton-brown.txt - The Wisdom of Father Brown,0.000518,chesterton-ball.txt - The Ball and The Cross,1.922207
6,7,"edgeworth-parents.txt - The Parent's Assistant,",0.000292,chesterton-brown.txt - The Wisdom of Father Brown,1.844820


... (7 outputs left out)

,Posicion,ID y titulo TF-IDF,Coseno,ID y titulo BM25,Score BM25
0,1,chesterton-brown.txt - The Wisdom of Father Brown,0.032501,chesterton-brown.txt - The Wisdom of Father Brown,6.816346
1,2,blake-poems.txt - Poems,0.010958,chesterton-thursday.txt - The Man Who Was Thur...,6.423723
2,3,chesterton-thursday.txt - The Man Who Was Thur...,0.008228,chesterton-ball.txt - The Ball and The Cross,5.194868
3,4,shakespeare-hamlet.txt - The Tragedie of Hamlet,0.006592,whitman-leaves.txt - Leaves of Grass,3.118189
4,5,bryant-stories.txt - Stories to Tell to Children,0.005964,melville-moby_dick.txt - Moby Dick,2.932929
5,6,"edgeworth-parents.txt - The Parent's Assistant,",0.005435,"edgeworth-parents.txt - The Parent's Assistant,",2.703815
6,7,austen-persuasion.txt - Persuasion,0.003830,carroll-alice.txt - Alice's Adventures in Wond...,2.643866
7,8,austen-emma.txt - Emma,0.003667,bible-kjv.txt - The King James Bible,2.177405
8,9,chesterton-ball.txt - The Ball and The Cross,0.003616,shakespeare-hamlet.txt - The Tragedie of Hamlet,1.606647
9,10,bible-kjv.txt - The King James Bible,0.003114,austen-sense.txt - Sense and Sensibility,1.254433



Consulta: zzzxqnotaword


,Posicion,ID y titulo TF-IDF,Coseno,ID y titulo BM25,Score BM25


,Consulta,Tamano TF-IDF,Tamano BM25,IDs compartidos,Primero TF-IDF,Primero BM25
0,the,10,10,8,whitman-leaves.txt,bible-kjv.txt
1,whale,6,6,6,melville-moby_dick.txt,melville-moby_dick.txt
2,alice rabbit,7,7,7,carroll-alice.txt,carroll-alice.txt
3,emma marriage love,10,10,7,austen-emma.txt,austen-emma.txt
4,king god israel,10,10,8,bible-kjv.txt,bible-kjv.txt
5,sea ship whale water,10,10,7,melville-moby_dick.txt,melville-moby_dick.txt
6,father brown detective murder mystery,10,10,6,chesterton-brown.txt,chesterton-brown.txt
7,zzzxqnotaword,0,0,0,(vacio),(vacio)


La comparación se hace por ID y posición. Los IDs compartidos son 8, 6, 7, 7, 8, 7, 6 y 0 sobre listas de tamaño 10, 6, 7, 10, 10, 10, 10 y 0.

En `whale` y `alice rabbit` las listas tienen menos de 10 libros, así que ambos métodos recuperan exactamente los mismos (6 de 6 y 7 de 7) y la única diferencia es el orden. 

En las consultas con 10 resultados se comparten entre 6 y 8 libros, de 2 a 4 libros de un top-10 quedan fuera del top-10 del otro método.

Para el primer documento coindicen los dos métodos en 6 de 7 consultas, difiriendo únicamente con `the` (*Whitman* en TF-IDF, la Biblia en BM25), que es la que no tiene ningún término discriminante. Cuando la consulta contiene términos raros (*emma*, *alice*, *whale*), los dos métodos coinciden en el mejor libro porque une TF alto y términos raros, los desacuerdos aparecen más abajo en la lista.

Luego, buscamos dos documentos que estando en dos rankings, tengan un orden invertido entre los métodos, donde si no se encuentra la inversión como tal, se selecciona un par donde la distancia entre posiciones cambie. Se guarda la consulta, los documentos y sus posiciones.

In [5]:
eleccion = None
alternativa = None
for caso in casos:
    posicion_tf = {}
    posicion_bm = {}
    for pos, i in enumerate(caso['orden_tfidf'], start=1):
        posicion_tf[i] = pos
    for pos, i in enumerate(caso['orden_bm25'], start=1):
        posicion_bm[i] = pos
    comunes = []
    for i in caso['orden_tfidf']:
        if i in posicion_bm:
            comunes.append(i)
    for a in range(len(comunes)):
        for b in range(a + 1, len(comunes)):
            i = comunes[a]
            j = comunes[b]
            distancia_tf = posicion_tf[i] - posicion_tf[j]
            distancia_bm = posicion_bm[i] - posicion_bm[j]
            if distancia_tf * distancia_bm < 0:
                eleccion = (caso, i, j, posicion_tf, posicion_bm)
                break
            if abs(distancia_tf) != abs(distancia_bm):
                alternativa = (caso, i, j, posicion_tf, posicion_bm)
        if eleccion is not None:
            break
    if eleccion is not None:
        break

if eleccion is None:
    eleccion = alternativa
if eleccion is None:
    raise ValueError('Sin cambio de posiciones')

Luego, para los dos documentos elegidos se muestra la longitud, posiciones y puntuaciones en cada metodo. Así entonces, se descompone la consulta término a término, presentando la frecuencia en el documento, el DF en el corpus y el aporte que cada termino hace al coseno y al BM25. 

In [6]:
caso_analisis, i, j, pos_tf, pos_bm = eleccion
norma_q = float(np.sqrt(caso_analisis['q_tfidf'].multiply(caso_analisis['q_tfidf']).sum()))
filas_documentos = []
filas_aportes = []
for indice in [i, j]:
    doc_id = identificadores[indice]
    fila_doc = matriz_tfidf.getrow(indice)
    norma_doc = float(np.sqrt(fila_doc.multiply(fila_doc).sum()))
    filas_documentos.append({
        'ID': doc_id, 'Titulo': filas_corpus[indice]['Titulo'],
        '|d|': longitudes[doc_id], 'Pos TF-IDF': pos_tf[indice],
        'Coseno': float(caso_analisis['cosenos'][indice]),
        'Pos BM25': pos_bm[indice], 'BM25': float(caso_analisis['bm25'][indice]),
    })
    vistos = set()
    for termino in caso_analisis['tokens']:
        if termino in vistos:
            continue
        vistos.add(termino)
        f = tf_por_documento[doc_id][termino]
        aporte_coseno = 0.0
        aporte_bm = 0.0
        if termino in columna_de:
            col = columna_de[termino]
            producto = float(caso_analisis['q_tfidf'][0, col] * fila_doc[0, col])
            if norma_q * norma_doc > 0:
                aporte_coseno = producto / (norma_q * norma_doc)
            aporte_bm = aporte_bm25(termino, doc_id, k1_base, b_base)
        filas_aportes.append({
            'ID': doc_id, 'Termino': termino, '|d|': longitudes[doc_id],
            'f(t,d)': f, 'df(t)': df_por_termino[termino],
            'Aporte coseno': aporte_coseno, 'Aporte BM25': aporte_bm,
        })

print('Consulta analizada:', caso_analisis['consulta'])
display(pd.DataFrame(filas_documentos).round(6))
display(pd.DataFrame(filas_aportes).round(6))

Consulta analizada: the


,ID,Titulo,|d|,Pos TF-IDF,Coseno,Pos BM25,BM25
0,whitman-leaves.txt,Leaves of Grass,124335,1,0.729540,2,0.058663
1,bible-kjv.txt,The King James Bible,852256,6,0.606733,1,0.058664


,ID,Termino,|d|,"f(t,d)",df(t),Aporte coseno,Aporte BM25
0,whitman-leaves.txt,the,124335,10113,18,0.729540,0.058663
1,bible-kjv.txt,the,852256,64023,18,0.606733,0.058664


Para la consulta *the*, TF-IDF + coseno coloca primero a *Whitman* y sexto a la Biblia. *Whitman* obtiene mayor similitud porque el peso de *the* en relación con su vector TF-IDF completo produce un coseno mayor.

BM25 ajusta el aporte de *the* según la longitud de cada libro y la frecuencia del término. La Biblia es mucho más larga, pero también contiene más apariciones de *the*. Tras aplicar ambos efectos, obtiene 0,058664 frente a 0,058663 de Whitman y pasa a la primera posición. La diferencia es mínima, ya que el cambio de orden es real pero las puntuaciones son casi iguales.

Como *the* aparece en los 18 libros, su IDF en BM25 es pequeño pero positivo. Por eso todos reciben un aporte parecido, entre 0,058623 y 0,058664. En TF-IDF + coseno, el término también participa en el cálculo, pero en cambio las similitudes varían más porque interviene la normalización del vector completo de cada documento. Así se tiene que BM25 no ignora *the*, si no que le asigna poco poder para distinguir entre libros.

In [7]:
def desglose_par(consulta, doc_a, doc_b):
    caso = None
    for c in casos:
        if c['consulta'] == consulta:
            caso = c
    pos_tf = {}
    pos_bm = {}
    for pos, i in enumerate(caso['orden_tfidf'], start=1):
        pos_tf[i] = pos
    for pos, i in enumerate(caso['orden_bm25'], start=1):
        pos_bm[i] = pos
    filas_doc = []
    filas_term = []
    for doc_id in [doc_a, doc_b]:
        i = identificadores.index(doc_id)
        fila_d = matriz_tfidf.getrow(i)
        norma_d = float(np.sqrt(fila_d.multiply(fila_d).sum()))
        K = 1 - b_base + b_base * longitudes[doc_id] / longitud_media
        filas_doc.append({
            'ID': doc_id, '|d|': longitudes[doc_id], 'K': K, 'Norma TF-IDF': norma_d,
            'Pos TF-IDF': pos_tf[i], 'Coseno': float(caso['cosenos'][i]),
            'Pos BM25': pos_bm[i], 'BM25': float(caso['bm25'][i]),
        })
        vistos = set()
        for termino in caso['tokens']:
            if termino in vistos:
                continue
            vistos.add(termino)
            f = tf_por_documento[doc_id][termino]
            saturacion = 0.0
            if f > 0:
                saturacion = f / (f + k1_base * K)
            filas_term.append({
                'ID': doc_id, 'Termino': termino, 'f(t,d)': f,
                'df(t)': df_por_termino[termino], 'f/(f+k1*K)': saturacion,
                'Aporte BM25': aporte_bm25(termino, doc_id, k1_base, b_base),
            })
    print('\nConsulta:', consulta)
    display(pd.DataFrame(filas_doc).round(6))
    display(pd.DataFrame(filas_term).round(6))

desglose_par('whale', 'shakespeare-hamlet.txt', 'whitman-leaves.txt')
desglose_par('alice rabbit', 'chesterton-thursday.txt', 'burgess-busterbrown.txt')
desglose_par('emma marriage love', 'austen-persuasion.txt', 'blake-poems.txt')


Consulta: whale


,ID,|d|,K,Norma TF-IDF,Pos TF-IDF,Coseno,Pos BM25,BM25
0,shakespeare-hamlet.txt,29698,0.433738,2726.165768,3,0.001540,4,1.872497
1,whitman-leaves.txt,124335,1.019245,13862.152459,4,0.000757,3,1.896003


,ID,Termino,"f(t,d)",df(t),f/(f+k1*K),Aporte BM25
0,shakespeare-hamlet.txt,whale,2,6,0.793498,1.872497
1,whitman-leaves.txt,whale,5,6,0.803459,1.896003



Consulta: alice rabbit


,ID,|d|,K,Norma TF-IDF,Pos TF-IDF,Coseno,Pos BM25,BM25
0,chesterton-thursday.txt,58300,0.610695,6119.758949,5,0.000571,2,3.509562
1,burgess-busterbrown.txt,15969,0.348798,1818.060225,2,0.007630,3,2.273300


,ID,Termino,"f(t,d)",df(t),f/(f+k1*K),Aporte BM25
0,chesterton-thursday.txt,alice,1,3,0.577089,2.147746
1,chesterton-thursday.txt,rabbit,1,6,0.577089,1.361816
2,burgess-busterbrown.txt,alice,0,3,0.000000,0.000000
3,burgess-busterbrown.txt,rabbit,11,6,0.963344,2.273300



Consulta: emma marriage love


,ID,|d|,K,Norma TF-IDF,Pos TF-IDF,Coseno,Pos BM25,BM25
0,austen-persuasion.txt,83659,0.767588,8000.437973,6,0.004145,2,3.651662
1,blake-poems.txt,6839,0.292312,713.299880,2,0.016165,7,1.124516


,ID,Termino,"f(t,d)",df(t),f/(f+k1*K),Aporte BM25
0,austen-persuasion.txt,emma,1,2,0.520534,2.322583
1,austen-persuasion.txt,marriage,27,12,0.967010,0.890774
2,austen-persuasion.txt,love,42,15,0.978540,0.438305
3,blake-poems.txt,emma,0,2,0.000000,0.000000
4,blake-poems.txt,marriage,1,12,0.740316,0.681952
5,blake-poems.txt,love,29,15,0.988049,0.442565


Dado que el par `the` tiene una diferencia numérica mínima en BM25, se añadieron tres pares donde la diferencia entre métodos es más visible.

- *whale* tiene df = 6 y aparece 2 veces en *Hamlet* y 5 en *Whitman*. El coseno en este caso prefiere a *Hamlet* que es mucho más corto, mientras que BM25 prefiere a *Whitman* que tiene más menciones. La diferencia en BM25 es pequeña, de un 1.3% aprox. 

- Para *alice rabbit*, *Thursday* contiene los dos términos, aunque solo una vez cada uno: aporta 2.1477 por *alice* (df = 3, el más raro) y 1.3618 por *rabbit*, en total 3.5096. *Buster Bear* solo tiene *rabbit* y llega a 2.2733. BM25 suma contribuciones por término y premia que el libro cubra los dos, sobre todo el raro. El coseno, en cambio premia la concentración, donde la norma de *Buster Bear* es 1,818 frente a 6,120 de *Thursday*, teniendo que su coseno resulta mayor.

- Para *emma marriage love*, *Blake* es el libro más corto, teniendo que su coseno sube a 0.0162 porque *love* ocupa una gran parte de un vector muy pequeño. *Persuasion* suma tres términos: *emma* (una sola vez), *marriage* (0.8908) y *love* (0.4383), haciendo un total 3.6517, frente a los 1.1245 de *Blake* (*marriage* 0.6820 + *love* 0.4426). Aquí pesa la rareza, donde una sola aparición de *emma* aporta más que todas las apariciones de *love* en *Blake* porque el IDF de *love* es mucho menor que el de *emma*.

En estos resultados, TF-IDF + coseno mide la similitud entre los vectores de la consulta y del libro, teniendo que la normalización influye en la posición de cada documento. También se tiene que BM25 suma los aportes de los términos según su frecuencia, su rareza y la longitud del libro, limitando el efecto de las repeticiones con la saturación. Por esas diferencias los dos métodos pueden recuperar los mismos libros y ordenarlos de una forma distinta.

**4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.


Se usó la consulta `the` para observar el efecto de los parámetros de BM25 en libros de longitudes y frecuencias diferentes. También se selecciona el libro más corto, el más largo y el que más repite el término, luego se compara sus posiciones y aportes con los parámetros base, utilizando $b=0$ y $k_1=2$.


In [8]:
consulta_parametros = 'the'
tokens_parametros = normalizar_tokenizar(consulta_parametros)
termino_parametro = tokens_parametros[0]
documentos_elegidos = []
mas_corto = identificadores[0]
mas_largo = identificadores[0]
for doc_id in identificadores:
    if longitudes[doc_id] < longitudes[mas_corto]:
        mas_corto = doc_id
    if longitudes[doc_id] > longitudes[mas_largo]:
        mas_largo = doc_id
documentos_elegidos.append(mas_corto)
documentos_elegidos.append(mas_largo)
mas_repeticiones = identificadores[0]
for doc_id in identificadores:
    frecuencia = tf_por_documento[doc_id][termino_parametro]
    if frecuencia > tf_por_documento[mas_repeticiones][termino_parametro]:
        mas_repeticiones = doc_id
if mas_repeticiones not in documentos_elegidos:
    documentos_elegidos.append(mas_repeticiones)

configuraciones = [
    ('Base', 1.2, 0.75),
    ('Sin ajuste por longitud', 1.2, 0.0),
    ('Mayor efecto de repeticiones', 2.0, 0.75),
]
filas_parametros = []
for nombre, k1, b in configuraciones:
    scores = scores_bm25(tokens_parametros, k1, b)
    orden = ordenar_positivos(scores)
    posiciones = {}
    for posicion, i in enumerate(orden, start=1):
        posiciones[identificadores[i]] = posicion
    for doc_id in documentos_elegidos:
        i = identificadores.index(doc_id)
        filas_parametros.append({
            'Ejecucion': nombre, 'k1': k1, 'b': b, 'ID': doc_id,
            '|d|': longitudes[doc_id], 'f(t,d)': tf_por_documento[doc_id][termino_parametro],
            'Posicion': posiciones.get(doc_id), 'Score BM25': float(scores[i]),
            'Aporte de the': aporte_bm25(termino_parametro, doc_id, k1, b),
        })

print('Consulta:', consulta_parametros, 'df:', df_por_termino[termino_parametro])
print('Longitud media:', round(longitud_media, 2))
display(pd.DataFrame(filas_parametros).round(6))
for doc_id in documentos_elegidos:
    datos = []
    for fila in filas_parametros:
        if fila['ID'] == doc_id:
            datos.append(fila)
    base, sin_longitud, mas_tf = datos
    print(doc_id, 'base:', base['Posicion'],
          'b=0:', sin_longitud['Posicion'], 'k1=2:', mas_tf['Posicion'])
    if base['Posicion'] == sin_longitud['Posicion'] == mas_tf['Posicion']:
        print('  La posicion sigue igual; los aportes de la tabla muestran el cambio.')
print('b=0 quita el ajuste por longitud; k1=2 aumenta el efecto de las repeticiones.')

Consulta: the df: 18
Longitud media: 121224.33


,Ejecucion,k1,b,ID,|d|,"f(t,d)",Posicion,Score BM25,Aporte de the
0,Base,1.2,0.75,blake-poems.txt,6839,439,18,0.058623,0.058623
1,Base,1.2,0.75,bible-kjv.txt,852256,64023,1,0.058664,0.058664
2,Sin ajuste por longitud,1.2,0.00,blake-poems.txt,6839,439,18,0.058510,0.058510
3,Sin ajuste por longitud,1.2,0.00,bible-kjv.txt,852256,64023,1,0.058669,0.058669
4,Mayor efecto de repeticiones,2.0,0.75,blake-poems.txt,6839,439,18,0.079898,0.079898
5,Mayor efecto de repeticiones,2.0,0.75,bible-kjv.txt,852256,64023,1,0.079991,0.079991


blake-poems.txt base: 18 b=0: 18 k1=2: 18
  La posicion sigue igual; los aportes de la tabla muestran el cambio.
bible-kjv.txt base: 1 b=0: 1 k1=2: 1
  La posicion sigue igual; los aportes de la tabla muestran el cambio.
b=0 quita el ajuste por longitud; k1=2 aumenta el efecto de las repeticiones.


Al aumentar $k1$ de 1,2 a 2, el aporte de *the* aumenta y las repeticiones tardan más en saturarse dado que el aumento no es idéntico para todos los libros. Al poner $b=0$ se elimina el ajuste por longitud, así en los documentos mostrados *Blake* pasa de 0,058623 a 0,058510 y la Biblia de 0,058664 a 0,058669.

En las tres configuraciones, los dos documentos seleccionados mantienen sus posiciones. Se tiene que *Blake* permanece en la 18 y la Biblia en la 1 a pesar de que sus aportes cambian. Con una consulta de un solo término y $b$ fijo, cambiar $k1$ modifica las puntuaciones sin alterar el orden, esto no implica que multiplique todas las puntuaciones por un mismo valor, si no que al modificar $b$ se cambia el ajuste por longitud, por lo que el orden de otros documentos sí podría cambiar.